## **Problem Set 2**. Neural networks & optimisation
### *Goal: derive the deep-learning backbone once, so it can be reused for text, images and audio.*

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

##### 1. **[Theory]** (Softmax + cross-entropy.) For logits $z \in \mathbb{R}^K$, softmax $p_k = e^{z_k}/\sum_j e^{z_j}$ and one-hot target $y$, show that the cross-entropy loss $L = − \sum_k y_k \log{p_k}$ has gradient $\frac{\partial{L}}{\partial z} = p − y$. Explain why this clean form is the reason softmax and cross-entropy are always paired.


$z \in \mathbb{R}^K$, softmax $p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$  

cross-entropy loss $L = - \sum_k y_k \log p_k$

Show that: $\frac{\partial L}{\partial z} = p - y$.

1. **Derivation of softmax and its derivative**

To get the derivative of the cross-entropy loss, we need the softmax and its derivative.

Let's start with the logarithm of $p_k$:
$$\log p_k = \log \left(\frac{e^{z_k}}{\sum_{j=1}^C e^{z_j}}\right) = \log(e^{z_k}) - \log\left(\sum_{j=1}^C e^{z_j}\right) = z_k - \log\left(\sum_{j=1}^C e^{z_j}\right)$$

Now, let's differentiate with respect to $z_i$:
$$\frac{\partial \log p_k}{\partial z_i} = \frac{\partial z_k}{\partial z_i} - \frac{\partial \log\left(\sum_j e^{z_j}\right)}{\partial z_i}$$

**1.1.** First term:
$$\frac{\partial z_k}{\partial z_i} = \begin{cases} 1 & \text{if } z_k = z_i \\ 0 & \text{otherwise} \end{cases} = \mathbb{1}(z_k = z_i) = \delta_{ki}$$

**1.2.** Second term:
$$\frac{\partial \log\left(\sum_{j=1}^C e^{z_j}\right)}{\partial z_i} = \frac{1}{\sum_{j=1}^C e^{z_j}} \cdot \frac{\partial \left(\sum_{j=1}^C e^{z_j}\right)}{\partial z_i} = \frac{1}{\sum_{j=1}^C e^{z_j}} \cdot \sum_{j=1}^C \frac{\partial e^{z_j}}{\partial z_i} =$$
$$= \frac{1}{\sum_{j=1}^C e^{z_j}} \cdot \sum_{j=1}^C e^{z_j} \frac{\partial z_j}{\partial z_i} = \frac{1}{\sum_{j=1}^C e^{z_j}} \cdot \sum_{j=1}^C e^{z_j} \delta_{ji} = \frac{e^{z_i}}{\sum_{j=1}^C e^{z_j}} = p_i$$

So finally:
$$\frac{\partial \log p_k}{\partial z_i} = \delta_{ki} - p_i$$

Since $\dfrac{\partial \log p_k}{\partial z_i} = \dfrac{1}{p_k} \cdot \dfrac{\partial p_k}{\partial z_i}$, we have:
$$\dfrac{\partial p_k}{\partial z_i} = p_k (\delta_{ki} - p_i)$$

Now we can use $\dfrac{\partial p_k}{\partial z_i}$ to write the Jacobian matrix:
$$\frac{\partial p}{\partial z} = \begin{bmatrix} 
\frac{\partial p_1}{\partial z_1} & \frac{\partial p_1}{\partial z_2} & \dots & \frac{\partial p_1}{\partial z_K} \\ 
\frac{\partial p_2}{\partial z_1} & \frac{\partial p_2}{\partial z_2} & \dots & \frac{\partial p_2}{\partial z_K}  \\ 
\vdots & \vdots & \ddots & \vdots \\ 
\frac{\partial p_K}{\partial z_1} & \frac{\partial p_K}{\partial z_2} & \dots & \frac{\partial p_K}{\partial z_K} 
\end{bmatrix} = \begin{bmatrix} 
p_1(1-p_1) & -p_1 p_2 & \dots & -p_1 p_K \\ 
-p_2 p_1 & p_2(1-p_2) & \dots & -p_2 p_K \\ 
\vdots & \vdots & \ddots & \vdots \\ 
-p_K p_1 & -p_K p_2 & \dots & p_K(1-p_K) 
\end{bmatrix}$$


2. **Differentiating cross-entropy loss**

$$L(p, y) = -\sum_{k} y_k \log p_k$$

$$\frac{\partial L(p, y)}{\partial z_i} = -\sum_{k=1}^C y_k \frac{\partial \log p_k}{\partial z_i} = -\sum_{k=1}^C \frac{y_k}{p_k} \frac{\partial p_k}{\partial z_i} =$$
$$= -\sum_{k=1}^C \frac{y_k}{p_k} p_k (\delta_{ki} - p_i) = -\sum_{k=1}^C y_k \delta_{ki} + \sum_{k=1}^C y_k p_i$$

Where $k = i$ only then the first term will become $y_i$:
$$\frac{\partial L(p, y)}{\partial z_i} = -y_i + \sum_{k=1}^C y_k p_i$$

Where $\sum_{k=1}^C y_k = 1$ as $y_i$ is a one-hot vector:
$$\frac{\partial L(p, y)}{\partial z_i} = -y_i + p_i$$

In the vectorized form it can be simply written as:
$$\frac{\partial L(p, y)}{\partial z} = p - y$$

$\blacksquare$

##### 2. **[Theory]** *(Backpropagation.)* Consider a two-layer MLP $h = \sigma(W_1 x + b_1)$, $\hat{y} = \text{softmax}(W_2 h + b_2)$ with cross-entropy loss. Derive $\partial L / \partial W_2$, $\partial L / \partial b_2$, $\partial L / \partial W_1$, $\partial L / \partial b_1$ explicitly using the chain rule, for a general elementwise activation $\sigma$.

##### 3. **[Theory]** Write the update rules of SGD, SGD+momentum and Adam. Show that momentum is an exponential moving average of gradients, and explain the purpose of Adam's bias-correction terms $\hat{m}_t, \hat{v}_t$ in the first few steps.

##### 4. **[Python]** Implement the two-layer MLP of Problem 2 in *pure NumPy*, including the backward pass. Verify every gradient by central finite differences (gradient check to $< 10^{-5}$ relative error).

[Multi-Layer Perceptron:](https://elcaiseri.medium.com/building-a-multi-layer-perceptron-from-scratch-with-numpy-e4cee82ab06d)

1. Layers and nodes:
- Input Layer: receives input data

- Hidden Layer: perform intermediate computations
- Output Layer: produces the final prediction

2. Activation functions:
- sigmoid $f(x) = \dfrac{1}{1+e^{-x}}$

- ReLU $R(z) = \max{(0,z)}$
- softmax $\text{softmax}(z_j) = \dfrac{e^{z_j}}{\sum_{k=1}^K e^{z_k}}$

3. Loss function:
- Cross-Entropy Loss: measures the performance of a classification model

4. Optimization:
- Gradient Descent: updates weights to minimize the loss function
- Backpropagation: computes gradients of the loss function with respect to each weight

[Backpropagation:](https://medium.com/@piyushkashyap045/understanding-multi-layer-perceptrons-and-backpropagation-in-neural-networks-7382d5b529d2) 

Method used to train the neural network by adjusting weights and biases based on the error of the prediction. It works by:

- Calculating the Error between the predicted and actual values.

- Adjusting Weights and Biases in each layer to minimize this error.
- Repeating the Process over multiple passes (called epochs) to improve prediction accuracy.

Backpropagation involves two steps:
1. Forward Propagation: Calculate the output based on current weights.
2. Backward Propagation: Use the error to update weights and biases by propagating backward through the layers.


In [11]:
class TwoLayerMLP:
    def __init__(self, input_size, hidden_size, output_size):
        self.w1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))

        self.w2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    def softmax(self, z):
        return np.exp(z)/(np.sum(np.exp(z), axis=1, keepdims=True))

    def forward(self, X):
        # First layer
        h = X @ self.w1 + self.b1
        H = self.sigmoid(h)

        # Output layer
        k = H @ self.w2 + self.b2
        P = self.softmax(k)

        return P, h, H

    def cross_entropy_loss(self, P, y):
        loss = -np.sum(y * np.log(P + 1e-8)) / y.shape[0]
        return loss

    def backward(self, X, y, h, H, P):
        # Output layer gradients
        dk = (P - y) / X.shape[0]
        dw2 = H.T @ dk
        db2 = np.sum(dk, axis=0, keepdims=True)

        # Hidden layer gradients
        dH = dk @ self.w2.T
        dh = dH * self.sigmoid_derivative(h)

        dw1 = X.T @ dh
        db1 = np.sum(dh, axis=0, keepdims=True)

        return {"w1": dw1, "b1": db1, "w2": dw2, "b2": db2}

    def numerical_gradient(self, X, y, param_name, eps=1e-5):
        param = getattr(self, param_name)
        grad = np.zeros_like(param)

        for i in range(param.shape[0]):
            for j in range(param.shape[1]):
                orig_value = param[i, j]

                # f(x + eps)
                param[i, j] = orig_value + eps
                P_plus, _, _ = self.forward(X)
                loss_plus = self.cross_entropy_loss(P_plus, y)

                # f(x - eps)
                param[i, j] = orig_value - eps
                P_minus, _, _ = self.forward(X)
                loss_minus = self.cross_entropy_loss(P_minus, y)

                # Central difference
                grad[i, j] = (loss_plus - loss_minus) / (2 * eps)

                param[i, j] = orig_value

        return grad

    def gradient_check(self, X, y):
        P, h, H = self.forward(X)
        analytical_gradients = self.backward(X, y, h, H, P)

        param_names = ["w1", "b1", "w2", "b2"]
        results = {}

        for name in param_names:
            num_grad = self.numerical_gradient(X, y, name)
            analytical_grad = analytical_gradients[name]

            numerator = np.linalg.norm(analytical_grad - num_grad)
            denominator = (np.linalg.norm(analytical_grad) + np.linalg.norm(num_grad) + 1e-12)

            relative_error = numerator / denominator
            results[name] = relative_error

        return results

In [19]:
np.random.seed(42)

X = np.array([
    [0.2, 0.5, 0.1, 0.7],
    [0.8, 0.3, 0.6, 0.2],
    [0.4, 0.9, 0.2, 0.5],
    [0.1, 0.6, 0.8, 0.3],
    [0.7, 0.2, 0.4, 0.9]
])

y = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
])

model = TwoLayerMLP(input_size=4, hidden_size=5, output_size=3)
errors = model.gradient_check(X, y)

results = pd.DataFrame([
    {"Parameter": name,
     "Relative error": error,
     "<1e-5": "Yes" if error < 1e-5 else "No"}
    for name, error in errors.items()
    ])

results


,Parameter,Relative error,<1e-5
0,w1,1.912831e-08,Yes
1,b1,2.120130e-08,Yes
2,w2,1.542027e-08,Yes
3,b2,1.535798e-08,Yes


##### 5. **[Python]** Re-implement the same network in PyTorch with **autograd**. Train it on the Problem-Set-1 feature set and reproduce the NumPy loss curve to confirm the implementations agree.

##### 6. **[Python]** Study regularisation: train with/without L2, dropout and early stopping; produce train/validation learning curves for each and a small table of final accuracies.

##### 7. **[Analysis]** Using the curves from Problem 6, identify where over- and under-fitting occur and argue which regulariser helped most on this dataset and why.